# Prompted Generation, Conditional Diffusion, and Modern Use

Diffusion models are often encountered not through unconditional sampling from a prior, but through prompting, editing, inpainting, outpainting, and other forms of controlled generation. The key question is therefore not only how a diffusion process is trained, but how that process becomes a *useful interface* for specifying what should be generated.

The emphasis here is more divulgative than in the mathematical chapters. The goal is to explain how probabilistic and algorithmic diffusion ideas become the user-facing behavior of modern image systems.


## Unconditional Versus Conditional Generation

In an unconditional model, we aim to learn a distribution $p(\boldsymbol{x})$. Sampling produces an image with no external control beyond the randomness of the latent seed. This is mathematically clean and extremely useful for understanding the core generative problem, but it offers limited user steering.

In a conditional model, we instead learn or approximate a distribution such as $p(\boldsymbol{x} | \boldsymbol{c})$, where $\boldsymbol{c}$ is side information. The conditioning variable may be a class label, a segmentation mask, another image, or a text prompt. Once conditioning is available, sampling becomes guided rather than free-form. The user no longer asks only for “a realistic image”; the user asks for “a realistic image consistent with this description.”


```{important}
The big conceptual jump from classical diffusion to modern text-to-image systems is **conditioning**. The model is no longer asked only to sample a plausible image, but to sample a plausible image that also matches an external request.
```


A simple example is the difference between sampling a random shoe from a trained fashion model and asking for "a red running shoe viewed from the side on a white background." The first task explores the marginal data distribution. The second explores a controlled slice of that distribution defined by a textual instruction. Modern image generation systems derive much of their usefulness from this second capability.


## Why Diffusion Became a Natural Home for Prompting

Diffusion models are especially compatible with conditioning because the denoiser already expects contextual information such as time. Extending that interface to additional context is conceptually natural. Instead of learning only
:::{math}
\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t, t),
:::
one learns a conditioned denoiser such as
:::{math}
\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t, t, \boldsymbol{c}),
:::
where $\boldsymbol{c}$ may encode text or another control signal.

The deep idea is that prompting does not require inventing a different generative mechanism. It changes what information the denoiser is allowed to consult while it performs each reverse step. That is why conditioned diffusion feels like a natural extension of unconditional diffusion rather than a separate technology.


In modern text-to-image systems, the prompt is first mapped into a representation by a language or text-embedding model. The diffusion denoiser then uses that representation while reversing noise. Architecturally, this is often implemented with **cross-attention**, meaning that image features at intermediate layers can attend to text features. The result is that the model does not merely know that "some text exists"; it can align different parts of the image-generation process with different semantic parts of the prompt.

## A High-Level Prompting Pipeline

A contemporary prompt-based diffusion pipeline is often easiest to understand in the following stages:

1. Encode the prompt into text features.
2. Sample initial Gaussian noise in pixel space or, more commonly, in a learned latent space.
3. Run the reverse denoising process while conditioning every step on the text representation.
4. Decode the final latent, if a latent diffusion model is used, back into image space.

The interface feels simple to the user, but internally the system is coordinating language understanding, image denoising, and often an autoencoding stage as well.

It is useful to emphasize that the prompt does not act like a single command injected at the very start and then forgotten. In a conditioned diffusion model, the prompt influences the denoising trajectory repeatedly. One can think of the model as repeatedly asking: given the current noisy image, and given the text description, in which direction should this sample be denoised next?

## Classifier-Free Guidance in Intuition

One of the most influential practical ideas is **classifier-free guidance**. The rough intuition is to train the model both with and without conditioning, then combine the two predictions during sampling. The unconditional branch says "move toward a plausible image." The conditional branch says "move toward an image matching the prompt." By amplifying the difference between them, one can increase prompt adherence.

This usually improves instruction following, but not for free. Stronger guidance can reduce diversity or create over-sharpened, less natural outputs. This tradeoff is one reason why prompting quality is never only about model size. It also depends on how conditional information is injected and how sampling is guided.

```{admonition} Example: One Prompt, Two Modes of Use
:class: numerical-example

Consider the prompt: "a chalk drawing of a bicycle leaning against a cafe wall at sunset."

In an unconditional generator trained on street scenes, there is no guarantee that sampling will ever produce this exact idea. It may generate a random street, a random bicycle, or no bicycle at all.

In a prompted diffusion model, the text representation biases every reverse denoising step toward an image that matches *chalk drawing*, *bicycle*, *cafe wall*, and *sunset*. The final image is still stochastic because the initial noise and sampler matter, but the randomness now lives inside a semantic corridor defined by the prompt rather than across the full image distribution.
```

## Prompted Generation Versus Image Editing

Prompting is not only for generation from scratch. The same conditional machinery can be used for **editing**, **inpainting**, **outpainting**, and **image-to-image translation**. In these settings, the condition is richer than pure text. It may include a source image, a mask indicating which pixels can change, or a structural guide such as edges or depth.

This is one reason diffusion became so widely adopted in creative tools. The denoising process can be constrained in many ways, which makes it suitable not only for invention, but also for controlled revision.

From a product perspective, this matters enormously. Many users do not want a random impressive picture. They want a specific poster refined, a background replaced, a product shot cleaned up, or a diagram restyled while preserving content. Prompted and conditioned diffusion systems are attractive precisely because they make these workflows possible within one shared probabilistic framework.

## How This Relates to Popular AI Assistants Today

As of **May 22, 2026**, prompt-based image generation is publicly exposed in **ChatGPT** through its image-generation interface. In that product experience, a user can ask for a new image in natural language, refine the request conversationally, and edit existing images. At a high level, this is the public-facing form of conditional image generation: a language interface is used to specify the condition, and the image model produces outputs aligned with that condition.

It is also useful to contrast this with **Claude**. Public Anthropic documentation currently describes Claude as supporting text output with image understanding and file creation workflows, but not as a general text-to-image chat system in the same way. This contrast separates two ideas that are often conflated in public discussion: a multimodal assistant can *understand* images without necessarily being an image *generator* in the product sense.


The important takeaway is the **interface shift**. Many people now encounter generative image models not by writing training code or sampling unconditional priors, but by chatting with a system and iteratively refining prompts. The visible face of image generation has therefore become strongly **conditional**, **interactive**, and **language-mediated**.


## Why Latent Diffusion Matters So Much in Practice

Earlier we introduced latent diffusion mainly as a computational strategy. In the context of prompt-based systems, its importance becomes even clearer. High-resolution image generation is expensive in pixel space. If a good autoencoder provides a compressed latent representation, then denoising and conditioning can happen in that smaller space while still preserving semantics well enough for the final decoded image to look detailed.

This design is one of the reasons modern prompt-based generation can be both expressive and computationally tractable. The language-conditioned reverse process happens where computation is cheaper, and the decoder reconstructs the final image afterward.

## A Healthy Public-Facing Interpretation

From a divulgative point of view, one can summarize modern diffusion image systems in a sentence: they start from noise, repeatedly remove uncertainty, and use the prompt as a semantic guide during that cleanup. This sentence is not the full mathematics, but it is close enough to the truth to be educational without being misleading.

At the same time, one should resist two oversimplifications. First, the prompt is not a magical spell that deterministically specifies one image; randomness and sampling design still matter. Second, the model is not "drawing like a human artist" in any direct psychological sense; it is following a learned high-dimensional denoising rule conditioned on text and data.

## Final Perspective

The public story of modern image generation is often told as if the main innovation were simply “models that understand prompts.” The deeper story is more structural. A diffusion model already knows how to refine noise into image structure through many small denoising decisions. Prompting works because each of those decisions can be conditioned on side information.

Prompted generation is therefore not a decorative interface added after the mathematics. It is a direct consequence of turning unconditional denoisers into conditional denoisers and then choosing practical guidance mechanisms that make the condition matter strongly enough at sampling time.
